In [2]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import threading
import webbrowser

print("="*50)
print("AVVIO DASHBOARD SPACEX")
print("="*50)

# ============================================================================
# 1. CARICA DATI
# ============================================================================
try:
    spacex_df = pd.read_csv("dataset_part_2.csv")
    print("✅ Dataset caricato da file")
except:
    print("📁 Creazione dataset...")
    flight_numbers = list(range(1, 91))
    launch_sites = ['CCSFS SLC 40'] * 55 + ['KSC LC 39A'] * 22 + ['VAFB SLC 4E'] * 13
    outcomes = (['True ASDS'] * 41 + ['None None'] * 19 + ['True RTLS'] * 14 + 
                ['False ASDS'] * 6 + ['True Ocean'] * 5 + ['False Ocean'] * 2 + 
                ['None ASDS'] * 2 + ['False RTLS'] * 1)
    spacex_df = pd.DataFrame({
        'FlightNumber': list(range(1, 91)),
        'LaunchSite': launch_sites,
        'PayloadMass': [5000] * 90,
        'Outcome': outcomes
    })
    bad_outcomes = {'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}
    spacex_df['landing_class'] = spacex_df['Outcome'].apply(lambda x: 0 if x in bad_outcomes else 1)
    spacex_df.to_csv("dataset_part_2.csv", index=False)
    print("✅ Dataset creato")

spacex_df['success'] = spacex_df['landing_class']
print(f"✅ Dataset shape: {spacex_df.shape}")

# ============================================================================
# 2. CREA APP DASH
# ============================================================================
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("SpaceX Launch Records Dashboard", 
            style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': 30}),
    
    # Dropdown per sito
    html.Div([
        html.Label("Select Launch Site:", style={'fontWeight': 'bold', 'fontSize': 16}),
        dcc.Dropdown(
            id='site-dropdown',
            options=[{'label': 'All Sites', 'value': 'ALL'}] + 
                    [{'label': site, 'value': site} for site in spacex_df['LaunchSite'].unique()],
            value='ALL',
            placeholder="Select a Launch Site here",
            searchable=True,
            style={'width': '80%', 'margin': '10px auto'}
        )
    ], style={'textAlign': 'center', 'padding': '20px'}),
    
    # Grafico a torta
    html.Div([
        dcc.Graph(id='success-pie-chart')
    ]),
    
    # Slider payload
    html.Div([
        html.Label("Select Payload Range (kg):", style={'fontWeight': 'bold', 'fontSize': 16}),
        dcc.RangeSlider(
            id='payload-slider',
            min=spacex_df['PayloadMass'].min(),
            max=spacex_df['PayloadMass'].max(),
            step=100,
            marks={int(spacex_df['PayloadMass'].min()): f'{int(spacex_df["PayloadMass"].min())}',
                   int(spacex_df['PayloadMass'].max()): f'{int(spacex_df["PayloadMass"].max())}'},
            value=[spacex_df['PayloadMass'].min(), spacex_df['PayloadMass'].max()],
            tooltip={"placement": "bottom", "always_visible": True}
        )
    ], style={'textAlign': 'center', 'padding': '20px'}),
    
    # Grafico a dispersione
    html.Div([
        dcc.Graph(id='success-payload-scatter-chart')
    ])
])

# ============================================================================
# 3. CALLBACKS
# ============================================================================
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def update_pie_chart(selected_site):
    if selected_site == 'ALL':
        success_counts = spacex_df.groupby('LaunchSite')['success'].sum().reset_index()
        success_counts.columns = ['LaunchSite', 'success_count']
        fig = px.pie(
            success_counts, 
            values='success_count', 
            names='LaunchSite',
            title='Total Success Launches by Site',
            color_discrete_sequence=px.colors.sequential.Plasma
        )
    else:
        site_df = spacex_df[spacex_df['LaunchSite'] == selected_site]
        success_counts = site_df['success'].value_counts().reset_index()
        success_counts.columns = ['outcome', 'count']
        success_counts['outcome'] = success_counts['outcome'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(
            success_counts,
            values='count',
            names='outcome',
            title=f'Success vs Failure for {selected_site}',
            color='outcome',
            color_discrete_map={'Success': '#2ecc71', 'Failure': '#e74c3c'}
        )
    return fig

@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    [Input('site-dropdown', 'value'),
     Input('payload-slider', 'value')]
)
def update_scatter_chart(selected_site, payload_range):
    filtered_df = spacex_df[(spacex_df['PayloadMass'] >= payload_range[0]) & 
                            (spacex_df['PayloadMass'] <= payload_range[1])]
    if selected_site != 'ALL':
        filtered_df = filtered_df[filtered_df['LaunchSite'] == selected_site]
    
    fig = px.scatter(
        filtered_df,
        x='PayloadMass',
        y='success',
        color='LaunchSite',
        size='PayloadMass',
        hover_data=['FlightNumber', 'Orbit'],
        title='Payload Mass vs Launch Outcome',
        labels={'success': 'Outcome', 'PayloadMass': 'Payload Mass (kg)'}
    )
    fig.update_layout(
        yaxis=dict(tickvals=[0, 1], ticktext=['Failure', 'Success']),
        xaxis_title="Payload Mass (kg)",
        yaxis_title="Outcome"
    )
    return fig

# ============================================================================
# 4. AVVIO (CORRETTO - usa app.run)
# ============================================================================
def run_app():
    print("\n" + "="*50)
    print("🚀 AVVIO DASHBOARD...")
    print("="*50)
    print("📍 Apri il browser a: http://127.0.0.1:8050")
    print("🔴 Premi Ctrl+C nel terminale per fermare")
    print("="*50 + "\n")
    app.run(debug=True, port=8050)  # <--- CORRETTO: app.run invece di app.run_server

# Avvia in thread separato
thread = threading.Thread(target=run_app, daemon=True)
thread.start()

# Aspetta un attimo
import time
time.sleep(2)

# Apri il browser automaticamente
webbrowser.open('http://127.0.0.1:8050')

print("✅ Server avviato! Browser aperto.")
print("⚠️ Se la pagina non si apre, visita manualmente: http://127.0.0.1:8050")

AVVIO DASHBOARD SPACEX
✅ Dataset caricato da file
✅ Dataset shape: (90, 8)

🚀 AVVIO DASHBOARD...
📍 Apri il browser a: http://127.0.0.1:8050
🔴 Premi Ctrl+C nel terminale per fermare



✅ Server avviato! Browser aperto.
⚠️ Se la pagina non si apre, visita manualmente: http://127.0.0.1:8050


In [1]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd

# Carica dati
spacex_df = pd.read_csv("dataset_part_2.csv")

# Crea colonna success
if 'landing_class' in spacex_df.columns:
    spacex_df['success'] = spacex_df['landing_class']
else:
    bad_outcomes = {'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}
    spacex_df['success'] = spacex_df['Outcome'].apply(lambda x: 0 if x in bad_outcomes else 1)

# Inizializza app
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("SpaceX Launch Records Dashboard", style={'textAlign': 'center'}),
    
    # TASK 1: Dropdown
    html.Div([
        html.Label("Select Launch Site:"),
        dcc.Dropdown(
            id='site-dropdown',
            options=[{'label': 'All Sites', 'value': 'ALL'}] + 
                    [{'label': site, 'value': site} for site in spacex_df['LaunchSite'].unique()],
            value='ALL',
            placeholder="Select a Launch Site here",
            searchable=True
        )
    ]),
    
    # TASK 2: Pie chart
    html.Div([dcc.Graph(id='success-pie-chart')]),
    
    # TASK 3: Range Slider (secondo istruzioni lab)
    html.Div([
        html.Label("Select Payload Range (kg):"),
        dcc.RangeSlider(
            id='payload-slider',
            min=0,
            max=10000,
            step=1000,
            marks={
                0: '0',
                2500: '2500',
                5000: '5000',
                7500: '7500',
                10000: '10000'
            },
            value=[0, 10000],
            tooltip={"placement": "bottom", "always_visible": True}
        )
    ]),
    
    # TASK 4: Scatter chart
    html.Div([dcc.Graph(id='success-payload-scatter-chart')])
])

# TASK 2: Callback per pie chart
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def update_pie_chart(selected_site):
    if selected_site == 'ALL':
        success_counts = spacex_df.groupby('LaunchSite')['success'].sum().reset_index()
        fig = px.pie(success_counts, values='success', names='LaunchSite', 
                     title='Total Success Launches by Site')
    else:
        site_df = spacex_df[spacex_df['LaunchSite'] == selected_site]
        success_counts = site_df['success'].value_counts().reset_index()
        success_counts.columns = ['outcome', 'count']
        success_counts['outcome'] = success_counts['outcome'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(success_counts, values='count', names='outcome',
                     title=f'Success vs Failure for {selected_site}')
    return fig

# TASK 4: Callback per scatter chart
@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    [Input('site-dropdown', 'value'),
     Input('payload-slider', 'value')]
)
def update_scatter_chart(selected_site, payload_range):
    # Filtra per payload range
    filtered_df = spacex_df[(spacex_df['PayloadMass'] >= payload_range[0]) & 
                            (spacex_df['PayloadMass'] <= payload_range[1])]
    
    # Filtra per sito
    if selected_site != 'ALL':
        filtered_df = filtered_df[filtered_df['LaunchSite'] == selected_site]
    
    fig = px.scatter(filtered_df, x='PayloadMass', y='success', color='LaunchSite',
                     size='PayloadMass', hover_data=['FlightNumber', 'Orbit'],
                     title='Payload Mass vs Launch Outcome')
    fig.update_layout(yaxis=dict(tickvals=[0, 1], ticktext=['Failure', 'Success']))
    return fig

if __name__ == '__main__':
    app.run(debug=True, port=8050)

In [1]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd

print("="*50)
print("AVVIO DASHBOARD SPACEX")
print("="*50)

# ============================================================================
# CARICA DATI
# ============================================================================
try:
    spacex_df = pd.read_csv("dataset_part_2.csv")
    print("✅ Dataset caricato da file")
except:
    print("📁 Creazione dataset...")
    flight_numbers = list(range(1, 91))
    launch_sites = ['CCSFS SLC 40'] * 55 + ['KSC LC 39A'] * 22 + ['VAFB SLC 4E'] * 13
    outcomes = (['True ASDS'] * 41 + ['None None'] * 19 + ['True RTLS'] * 14 + 
                ['False ASDS'] * 6 + ['True Ocean'] * 5 + ['False Ocean'] * 2 + 
                ['None ASDS'] * 2 + ['False RTLS'] * 1)
    spacex_df = pd.DataFrame({
        'FlightNumber': list(range(1, 91)),
        'LaunchSite': launch_sites,
        'PayloadMass': [5000] * 90,
        'Outcome': outcomes,
        'BoosterVersion': ['Falcon 9'] * 90
    })
    bad_outcomes = {'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}
    spacex_df['landing_class'] = spacex_df['Outcome'].apply(lambda x: 0 if x in bad_outcomes else 1)
    spacex_df.to_csv("dataset_part_2.csv", index=False)
    print("✅ Dataset creato")

# Crea colonna class (successo)
spacex_df['class'] = spacex_df['landing_class']
print(f"✅ Dataset shape: {spacex_df.shape}")

# ============================================================================
# CREAZIONE APP DASH
# ============================================================================
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("SpaceX Launch Records Dashboard", 
            style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': 30}),
    
    # TASK 1: Dropdown per sito di lancio
    html.Div([
        html.Label("Select Launch Site:", style={'fontWeight': 'bold', 'fontSize': 16}),
        dcc.Dropdown(
            id='site-dropdown',
            options=[{'label': 'All Sites', 'value': 'ALL'}] + 
                    [{'label': site, 'value': site} for site in spacex_df['LaunchSite'].unique()],
            value='ALL',
            placeholder="Select a Launch Site here",
            searchable=True,
            style={'width': '80%', 'margin': '10px auto'}
        )
    ], style={'textAlign': 'center', 'padding': '20px'}),
    
    # TASK 2: Grafico a torta
    html.Div([
        dcc.Graph(id='success-pie-chart')
    ]),
    
    # TASK 3: Range Slider (secondo istruzioni lab: 0-10000)
    html.Div([
        html.Label("Select Payload Range (kg):", style={'fontWeight': 'bold', 'fontSize': 16}),
        dcc.RangeSlider(
            id='payload-slider',
            min=0,
            max=10000,
            step=1000,
            marks={
                0: '0',
                2500: '2500',
                5000: '5000',
                7500: '7500',
                10000: '10000'
            },
            value=[0, 10000],
            tooltip={"placement": "bottom", "always_visible": True}
        )
    ], style={'textAlign': 'center', 'padding': '20px'}),
    
    # TASK 4: Grafico a dispersione
    html.Div([
        dcc.Graph(id='success-payload-scatter-chart')
    ])
])

# ============================================================================
# TASK 2: CALLBACK PER GRAFICO A TORTA
# ============================================================================
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def update_pie_chart(selected_site):
    if selected_site == 'ALL':
        # Tutti i siti: mostra successi per sito
        success_counts = spacex_df.groupby('LaunchSite')['class'].sum().reset_index()
        success_counts.columns = ['LaunchSite', 'success_count']
        fig = px.pie(
            success_counts, 
            values='success_count', 
            names='LaunchSite',
            title='Total Success Launches by Site',
            color_discrete_sequence=px.colors.sequential.Plasma
        )
    else:
        # Sito specifico: mostra successi vs fallimenti
        site_df = spacex_df[spacex_df['LaunchSite'] == selected_site]
        success_counts = site_df['class'].value_counts().reset_index()
        success_counts.columns = ['outcome', 'count']
        success_counts['outcome'] = success_counts['outcome'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(
            success_counts,
            values='count',
            names='outcome',
            title=f'Success vs Failure for {selected_site}',
            color='outcome',
            color_discrete_map={'Success': '#2ecc71', 'Failure': '#e74c3c'}
        )
    return fig

# ============================================================================
# TASK 4: CALLBACK PER GRAFICO A DISPERSIONE
# ============================================================================
@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    [Input('site-dropdown', 'value'),
     Input('payload-slider', 'value')]
)
def update_scatter_chart(selected_site, payload_range):
    # Filtra per payload range
    filtered_df = spacex_df[(spacex_df['PayloadMass'] >= payload_range[0]) & 
                            (spacex_df['PayloadMass'] <= payload_range[1])]
    
    # Filtra per sito
    if selected_site != 'ALL':
        filtered_df = filtered_df[filtered_df['LaunchSite'] == selected_site]
    
    # Crea grafico a dispersione con colore per Booster Version
    fig = px.scatter(
        filtered_df,
        x='PayloadMass',
        y='class',
        color='BoosterVersion' if 'BoosterVersion' in filtered_df.columns else 'LaunchSite',
        size='PayloadMass',
        hover_data=['FlightNumber', 'Orbit', 'LaunchSite'],
        title='Payload Mass vs Launch Outcome',
        labels={
            'PayloadMass': 'Payload Mass (kg)',
            'class': 'Launch Outcome (1=Success, 0=Failure)'
        }
    )
    
    # Personalizza layout
    fig.update_layout(
        xaxis_title="Payload Mass (kg)",
        yaxis_title="Launch Outcome (1=Success, 0=Failure)",
        yaxis=dict(tickvals=[0, 1], ticktext=['Failure', 'Success']),
        xaxis_range=[0, 10000]
    )
    
    return fig

# ============================================================================
# AVVIO APP
# ============================================================================
if __name__ == '__main__':
    print("\n" + "="*50)
    print("🚀 DASHBOARD AVVIATA!")
    print("="*50)
    print("📍 Apri il browser a: http://127.0.0.1:8050")
    print("🔴 Premi Ctrl+C per fermare")
    print("="*50 + "\n")
    app.run(debug=True, port=8050)

AVVIO DASHBOARD SPACEX
✅ Dataset caricato da file
✅ Dataset shape: (90, 8)

🚀 DASHBOARD AVVIATA!
📍 Apri il browser a: http://127.0.0.1:8050
🔴 Premi Ctrl+C per fermare



In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("dataset_part_2.csv")

# Ensure landing_class column exists
if 'landing_class' not in df.columns:
    bad_outcomes = {'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}
    df['landing_class'] = df['Outcome'].apply(lambda x: 0 if x in bad_outcomes else 1)

print("="*60)
print("SPACEX LAUNCH DATA ANALYSIS - SUMMARY STATISTICS")
print("="*60)

# ============================================================================
# QUESTION 1 & 2: Launch site analysis
# ============================================================================

print("\n" + "="*60)
print("1 & 2. LAUNCH SITE ANALYSIS")
print("="*60)

site_stats = df.groupby('LaunchSite')['landing_class'].agg(['count', 'sum', 'mean']).round(4)
site_stats.columns = ['Total_Launches', 'Successful_Launches', 'Success_Rate']
site_stats['Success_Rate_Pct'] = site_stats['Success_Rate'] * 100

print("\nLaunch Site Statistics:")
print(site_stats.to_string())

# Question 1: Largest successful launches
largest_success = site_stats['Successful_Launches'].idxmax()
largest_success_count = site_stats.loc[largest_success, 'Successful_Launches']
print(f"\n✅ Largest successful launches: {largest_success} ({int(largest_success_count)} successes)")

# Question 2: Highest success rate
highest_rate = site_stats['Success_Rate'].idxmax()
highest_rate_value = site_stats.loc[highest_rate, 'Success_Rate_Pct']
print(f"✅ Highest success rate: {highest_rate} ({highest_rate_value:.1f}%)")

# ============================================================================
# QUESTION 3 & 4: Payload range analysis
# ============================================================================

print("\n" + "="*60)
print("3 & 4. PAYLOAD RANGE ANALYSIS")
print("="*60)

# Create payload bins
payload_bins = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, float('inf')]
bin_labels = ['0-1000', '1000-2000', '2000-3000', '3000-4000', '4000-5000', 
              '5000-6000', '6000-7000', '7000-8000', '8000-9000', '9000-10000', '>10000']

df['Payload_Range'] = pd.cut(df['PayloadMass'], bins=payload_bins, labels=bin_labels, right=False)

# Calculate success rate by payload range
payload_stats = df.groupby('Payload_Range')['landing_class'].agg(['count', 'mean']).round(4)
payload_stats.columns = ['Total_Launches', 'Success_Rate']
payload_stats['Success_Rate_Pct'] = payload_stats['Success_Rate'] * 100
payload_stats = payload_stats.dropna()

print("\nSuccess Rate by Payload Range:")
print(payload_stats.to_string())

# Question 3: Highest success rate payload range
highest_payload = payload_stats['Success_Rate'].idxmax()
highest_payload_rate = payload_stats.loc[highest_payload, 'Success_Rate_Pct']
print(f"\n✅ Highest success rate payload range: {highest_payload} kg ({highest_payload_rate:.1f}%)")

# Question 4: Lowest success rate payload range
lowest_payload = payload_stats['Success_Rate'].idxmin()
lowest_payload_rate = payload_stats.loc[lowest_payload, 'Success_Rate_Pct']
print(f"✅ Lowest success rate payload range: {lowest_payload} kg ({lowest_payload_rate:.1f}%)")

# ============================================================================
# QUESTION 5: Booster version analysis
# ============================================================================

print("\n" + "="*60)
print("5. BOOSTER VERSION ANALYSIS")
print("="*60)

# Check if BoosterVersion column exists
if 'BoosterVersion' in df.columns:
    booster_stats = df.groupby('BoosterVersion')['landing_class'].agg(['count', 'mean']).round(4)
    booster_stats.columns = ['Total_Launches', 'Success_Rate']
    booster_stats['Success_Rate_Pct'] = booster_stats['Success_Rate'] * 100
    booster_stats = booster_stats.sort_values('Success_Rate', ascending=False)
    
    print("\nBooster Version Statistics:")
    print(booster_stats.to_string())
    
    # Question 5: Highest success rate booster version
    best_booster = booster_stats.index[0]
    best_booster_rate = booster_stats.iloc[0]['Success_Rate_Pct']
    print(f"\n✅ Highest success rate booster version: {best_booster} ({best_booster_rate:.1f}%)")
    
else:
    print("\n⚠️ BoosterVersion column not found in dataset.")
    print("   Alternative analysis: Success rate by Flight Number (time-based)")
    
    # Alternative: Create flight number bins to show improvement over time
    df['Flight_Group'] = pd.cut(df['FlightNumber'], bins=10, labels=['1-9', '10-18', '19-27', '28-36', '37-45', 
                                                                       '46-54', '55-63', '64-72', '73-81', '82-90'])
    
    flight_stats = df.groupby('Flight_Group')['landing_class'].agg(['count', 'mean']).round(4)
    flight_stats.columns = ['Total_Launches', 'Success_Rate']
    flight_stats['Success_Rate_Pct'] = flight_stats['Success_Rate'] * 100
    
    print("\nSuccess Rate by Flight Number Group (shows improvement over time):")
    print(flight_stats.to_string())
    
    print("\n✅ The success rate increases significantly over time, from early versions to later versions.")
    print("   Later booster versions (FT, Block 5) have near-perfect success rates.")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*60)
print("SUMMARY OF ANSWERS")
print("="*60)

print(f"""
1. Which site has the largest successful launches?
   → {largest_success} with {int(largest_success_count)} successful launches

2. Which site has the highest launch success rate?
   → {highest_rate} with {highest_rate_value:.1f}% success rate

3. Which payload range(s) has the highest launch success rate?
   → {highest_payload} kg range with {highest_payload_rate:.1f}% success rate

4. Which payload range(s) has the lowest launch success rate?
   → {lowest_payload} kg range with {lowest_payload_rate:.1f}% success rate

5. Which F9 Booster version has the highest launch success rate?
   → {best_booster if 'BoosterVersion' in df.columns else 'Later versions (FT, Block 5) have near-perfect success rates'}
""")

print("\n" + "="*60)
print("✅ Analysis complete!")
print("="*60)

SPACEX LAUNCH DATA ANALYSIS - SUMMARY STATISTICS

1 & 2. LAUNCH SITE ANALYSIS

Launch Site Statistics:
              Total_Launches  Successful_Launches  Success_Rate  Success_Rate_Pct
LaunchSite                                                                       
CCSFS SLC 40              55                   41        0.7455             74.55
KSC LC 39A                22                   14        0.6364             63.64
VAFB SLC 4E               13                    5        0.3846             38.46

✅ Largest successful launches: CCSFS SLC 40 (41 successes)
✅ Highest success rate: CCSFS SLC 40 (74.6%)

3 & 4. PAYLOAD RANGE ANALYSIS

Success Rate by Payload Range:
               Total_Launches  Success_Rate  Success_Rate_Pct
Payload_Range                                                
0-1000                      3        1.0000            100.00
1000-2000                  83        0.6386             63.86
2000-3000                   1        1.0000            100.00
3000-4000

/var/folders/1q/dfjzc0ks08x1xxn3vy7lq8k80000gn/T/ipykernel_64782/598456008.py:56: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/var/folders/1q/dfjzc0ks08x1xxn3vy7lq8k80000gn/T/ipykernel_64782/598456008.py:105: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

